# NPDP 模拟专利数据生成器

本 Notebook 生成可直接用于 NPDP 训练和测试的模拟数据。核心字段为 `title`、`abstract` 和取值位于 0–1 的 `target`。额外字段仅用于检查数据，不会影响模型训练。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
N_SAMPLES = 500
TRAIN_RATIO = 0.8
rng = np.random.default_rng(SEED)

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
data_dir = repo_root / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {data_dir.resolve()}')

## 生成规则

模拟标签由技术新颖性、应用范围、权利要求强度和商业化潜力共同决定，并加入随机噪声。文本会包含与这些潜在因素对应的描述，使模型能够在小规模演示中学习到非随机关系。

In [ ]:
fields = {
    'artificial intelligence': ['neural inference', 'multimodal learning', 'edge intelligence'],
    'semiconductor': ['chip packaging', 'power transistor', 'memory architecture'],
    'renewable energy': ['battery management', 'solar conversion', 'hydrogen storage'],
    'biotechnology': ['molecular detection', 'drug delivery', 'cell engineering'],
    'advanced manufacturing': ['precision machining', 'industrial robotics', 'additive manufacturing'],
}
novelty_words = ['incremental', 'improved', 'novel', 'breakthrough']
scope_words = ['specialized', 'domain-specific', 'general-purpose', 'cross-industry']

rows = []
for index in range(N_SAMPLES):
    field = rng.choice(list(fields))
    technology = rng.choice(fields[field])
    novelty = rng.beta(2.2, 2.0)
    scope = rng.beta(2.0, 2.2)
    claim_strength = rng.beta(2.5, 2.0)
    commercialization = rng.beta(2.0, 2.0)

    novelty_word = novelty_words[min(int(novelty * len(novelty_words)), len(novelty_words) - 1)]
    scope_word = scope_words[min(int(scope * len(scope_words)), len(scope_words) - 1)]
    efficiency_gain = int(5 + 45 * novelty)
    deployment_count = int(1 + 9 * commercialization)

    title = f'{novelty_word.title()} {technology.title()} System for {scope_word.title()} Applications'
    abstract = (
        f'A {novelty_word} patent in {field} is disclosed for {technology}. '
        f'The invention provides a {scope_word} architecture with an estimated '
        f'{efficiency_gain}% efficiency improvement. Its claims cover {deployment_count} '
        f'deployment scenarios and emphasize scalable implementation, technical reliability, '
        f'and commercialization potential.'
    )

    latent_value = (
        0.35 * novelty
        + 0.20 * scope
        + 0.25 * claim_strength
        + 0.20 * commercialization
        + rng.normal(0, 0.05)
    )
    target = float(np.clip(latent_value, 0.0, 1.0))
    rows.append({
        'patent_id': f'MOCK-{index + 1:05d}',
        'technology_field': field,
        'title': title,
        'abstract': abstract,
        'target': round(target, 6),
    })

data = pd.DataFrame(rows)
data.head()

In [ ]:
data[['target']].describe()

## 划分并保存

数据先随机打乱，再按 80%/20% 划分。生成的 CSV 可直接传给各模型的训练与测试命令。

In [ ]:
shuffled = data.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
split_index = int(len(shuffled) * TRAIN_RATIO)
train_data = shuffled.iloc[:split_index].reset_index(drop=True)
test_data = shuffled.iloc[split_index:].reset_index(drop=True)

train_path = data_dir / 'mock_patents_train.csv'
test_path = data_dir / 'mock_patents_test.csv'
full_path = data_dir / 'mock_patents_full.csv'

train_data.to_csv(train_path, index=False, encoding='utf-8-sig')
test_data.to_csv(test_path, index=False, encoding='utf-8-sig')
shuffled.to_csv(full_path, index=False, encoding='utf-8-sig')

print(f'Train: {len(train_data):,} rows -> {train_path}')
print(f'Test:  {len(test_data):,} rows -> {test_path}')
print(f'Full:  {len(shuffled):,} rows -> {full_path}')

In [ ]:
assert train_data[['title', 'abstract', 'target']].notna().all().all()
assert test_data[['title', 'abstract', 'target']].notna().all().all()
assert train_data['target'].between(0, 1).all()
assert test_data['target'].between(0, 1).all()
print('Validation passed: required columns are complete and targets are within [0, 1].')

## 训练示例

```bash
python -m qwen_for_npdp.finetune \
  --checkpoint Qwen/Qwen2.5-7B \
  --data_path data/mock_patents_train.csv \
  --label_column target \
  --load_in_4bit --bf16
```